| Method | On-Policy | Off-Policy |
|---|---|---|
| **Value-Based** | SARSA | Q-Learning, DQN |
| **Actor-Critic** | A2C, PPO | TD3, SAC |
| **Pure Policy Gradient** | REINFORCE | Off-Policy REINFORCE (with Importance Sampling Correction) |

假设 REINFORCE 每次采集 1,000 个 transitions，但只做一次梯度更新就丢掉数据。为什么我们希望在同一批数据上做多个 gradient updates？为什么这样做又会产生新问题？

REINFORCE:
collect data using current policy, update once, collect data using the new urrent policy, update once....

people say, no, too inefficient, we want collect many data using current policy, then update many time (once per each data trace we collect). Environment Interaction 太贵了!

核心是两个相互冲突的目标：
1. Sample Efficiency：与 Environment 交互很昂贵，我们希望同一批数据能够训练多次。
2. Gradient Correctness：每次 Policy 更新之后，旧数据就不再来自当前 Policy。如果直接重复计算 REINFORCE Gradient，估计可能出现 Bias。



In [1]:
import numpy as np

rng = np.random.default_rng(42)

old = np.array([0.8, 0.2])
new = np.array([0.5, 0.5])
rewards = np.array([0.0, 10.0])

actions = rng.choice(2, size=10_000, p=old)

ratio = new[actions] / old[actions]

print("Old batch average:", rewards[actions].mean())
print("IS estimate:", np.mean(ratio * rewards[actions]))
print("True new expectation:", new @ rewards)

Old batch average: 1.954
IS estimate: 4.885
True new expectation: 5.0


### Importance Sampling（IS）


神奇的地方来了：

我们没有从 New Policy 采样任何数据，但利用 Old Policy 的数据，几乎准确地估计了 New Policy 的 Expected Reward！

这就是 Importance Sampling。

想象 Old Policy 采集了 1,000 个 Actions。
